In [1]:
data = open('names.txt', 'r').read().splitlines(keepends=False)
print(data[:5])

['emma', 'olivia', 'ava', 'isabella', 'sophia']


In [3]:
from models import *

bigram = Bigram()
bigram.fit(data)

In [3]:
bigram.make(5)

['kyason.', 'kanana.', 'milaveela.', 'ra.', 'drisan.']

In [4]:
bigram.loss(data)

2.4543561935424805

In [37]:
xs, ys = [], []

stoi = {s: i+1 for i, s in enumerate(string.ascii_lowercase)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

for word in data[:1]:
    chars = ['.'] + list(word) + ['.']
    for x, y in zip(chars, chars[1:]):
        xs.append(stoi[x])
        ys.append(stoi[y])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
xs, ys

(tensor([ 0,  5, 13, 13,  1]), tensor([ 5, 13, 13,  1,  0]))

In [40]:
import torch.nn.functional as fun

xenc = fun.one_hot(xs, num_classes=27).float()

In [41]:
W = torch.tensor(torch.randn(27, 27), requires_grad=True) # 27 neurons in 1 layer, each with 27 inputs
W

/var/folders/b4/ntxz7w0d3qx_38l6cvkkzb0c0000gn/T/ipykernel_37738/445059293.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  W = torch.tensor(torch.randn(27, 27), requires_grad=True) # 27 neurons in 1 layer, each with 27 inputs


tensor([[-1.0570e-01,  1.6175e+00,  3.6488e-01,  1.7678e-02,  1.3886e+00,
          6.4814e-02, -1.9536e+00, -4.2378e-01, -1.7890e+00,  1.1956e+00,
          1.0770e+00, -2.4885e-01,  2.0964e+00,  8.7149e-02, -2.3984e+00,
          2.3668e+00,  4.4279e-01,  1.5345e+00, -2.8094e-01,  1.4459e+00,
         -1.7007e+00,  7.1027e-02,  2.7313e-01,  1.9882e+00,  5.7412e-02,
          9.8134e-02,  1.2799e-01],
        [-1.3275e+00, -1.4770e+00, -8.6287e-01,  1.8695e-01, -1.2835e+00,
         -8.9591e-01,  9.6549e-01, -6.7153e-01, -5.1761e-01, -6.5691e-01,
          1.2381e+00, -5.3985e-01, -1.0347e+00,  1.1724e-01,  7.0429e-01,
         -2.3306e+00, -2.3399e-01, -1.1128e+00, -1.0235e+00, -1.4142e+00,
         -1.2061e+00, -7.1176e-01,  8.6919e-01,  1.2583e+00,  8.8455e-01,
         -6.6317e-01, -1.9322e+00],
        [-5.0647e-01, -1.2095e-01,  1.3493e+00, -1.7412e+00, -8.4255e-01,
          4.1216e-01, -2.7401e-01,  7.5751e-02, -4.7660e-01, -7.9649e-01,
         -1.4387e+00,  7.1618e-02,  1.74

In [73]:
logits = xenc @ W # log counts
counts = logits.exp() # counts
probs = counts / counts.sum(dim=1, keepdim=True)
probs

tensor([[0.0141, 0.0756, 0.0224, 0.0159, 0.0607, 0.0324, 0.0022, 0.0103, 0.0026,
         0.0504, 0.0449, 0.0122, 0.1183, 0.0170, 0.0014, 0.1511, 0.0242, 0.0698,
         0.0118, 0.0641, 0.0029, 0.0168, 0.0205, 0.1070, 0.0165, 0.0172, 0.0177],
        [0.0127, 0.0052, 0.0535, 0.2367, 0.0206, 0.0118, 0.0206, 0.0088, 0.0155,
         0.0048, 0.0226, 0.0299, 0.0152, 0.0208, 0.0282, 0.1119, 0.0139, 0.0144,
         0.0385, 0.0173, 0.1164, 0.0252, 0.0253, 0.0210, 0.0105, 0.0522, 0.0469],
        [0.0736, 0.1352, 0.0087, 0.0423, 0.0326, 0.0340, 0.0814, 0.0102, 0.0718,
         0.0110, 0.0193, 0.0228, 0.0185, 0.0232, 0.0405, 0.0279, 0.0479, 0.0307,
         0.0496, 0.0160, 0.0195, 0.0114, 0.0305, 0.0585, 0.0110, 0.0435, 0.0285],
        [0.0736, 0.1352, 0.0087, 0.0423, 0.0326, 0.0340, 0.0814, 0.0102, 0.0718,
         0.0110, 0.0193, 0.0228, 0.0185, 0.0232, 0.0405, 0.0279, 0.0479, 0.0307,
         0.0496, 0.0160, 0.0195, 0.0114, 0.0305, 0.0585, 0.0110, 0.0435, 0.0285],
        [0.0204, 0.0090,

In [74]:
nlls = torch.zeros(5)

for i in range(5):
    xi = xs[i].item()
    yi = ys[i].item()

    prob_yi = probs[i, yi]
    log_likelihood = prob_yi.log()
    nll = -log_likelihood
    nlls[i] = nll

total_loss = nlls.mean()
print(f'{total_loss=}')

total_loss=tensor(3.3923, grad_fn=<MeanBackward0>)


In [75]:
total_loss.backward()

W.data += -0.1 * W.grad

In [83]:
Xs, ys = [], []
# prepare the dataset
for word in data:
    chars = ['.'] + list(word) + ['.']
    for x, y in zip(chars, chars[1:]):
        Xs.append(stoi[x])
        ys.append(stoi[y])

Xs = torch.tensor(Xs)
ys = torch.tensor(ys)
num = Xs.numel()
num

228146

In [107]:
generator = torch.Generator().manual_seed(1234)
W = torch.randn((27, 27), requires_grad=True, generator=generator)

In [108]:
learning_rate = 50
num_epochs = 200

# iterate for num_epoch times
for epoch in range(num_epochs):
    xenc = fun.one_hot(Xs, num_classes=27).float()
    # forward pass
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(dim=1, keepdim=True)

    # loss
    loss = -probs[torch.arange(num), ys].log().mean()
    print(f'Epoch {epoch + 1}: nll={loss:.4f}')
    # backward pass

    W.grad = None
    loss.backward()
    # update
    W.data += -learning_rate * W.grad

Epoch 1: nll=3.8109
Epoch 2: nll=3.4293
Epoch 3: nll=3.1997
Epoch 4: nll=3.0405
Epoch 5: nll=2.9279
Epoch 6: nll=2.8480
Epoch 7: nll=2.7898
Epoch 8: nll=2.7460
Epoch 9: nll=2.7121
Epoch 10: nll=2.6852
Epoch 11: nll=2.6634
Epoch 12: nll=2.6454
Epoch 13: nll=2.6302
Epoch 14: nll=2.6173
Epoch 15: nll=2.6061
Epoch 16: nll=2.5963
Epoch 17: nll=2.5877
Epoch 18: nll=2.5800
Epoch 19: nll=2.5731
Epoch 20: nll=2.5669
Epoch 21: nll=2.5612
Epoch 22: nll=2.5561
Epoch 23: nll=2.5514
Epoch 24: nll=2.5471
Epoch 25: nll=2.5432
Epoch 26: nll=2.5395
Epoch 27: nll=2.5361
Epoch 28: nll=2.5330
Epoch 29: nll=2.5301
Epoch 30: nll=2.5273
Epoch 31: nll=2.5248
Epoch 32: nll=2.5224
Epoch 33: nll=2.5202
Epoch 34: nll=2.5181
Epoch 35: nll=2.5161
Epoch 36: nll=2.5142
Epoch 37: nll=2.5125
Epoch 38: nll=2.5108
Epoch 39: nll=2.5092
Epoch 40: nll=2.5077
Epoch 41: nll=2.5063
Epoch 42: nll=2.5050
Epoch 43: nll=2.5037
Epoch 44: nll=2.5024
Epoch 45: nll=2.5013
Epoch 46: nll=2.5002
Epoch 47: nll=2.4991
Epoch 48: nll=2.4981
E

In [114]:
generator = torch.Generator().manual_seed(1234)
results = []
for _ in range(5):
    out = []
    idx = 0
    while True:
        idx_enc = fun.one_hot(torch.tensor([idx]), num_classes=27).float()
        logits = idx_enc @ W
        counts = logits.exp()
        probs = counts / counts.sum(dim=1, keepdim=True)

        idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=generator).item()
        out.append(itos[idx])

        if idx == 0:
            break

    results.append(''.join(out))

results

['kyason.', 'kanana.', 'milaveela.', 'ra.', 'drisan.']